# W07_M2.2_轴承寿命预测系统 —— 教师版·分步执行（ABCD 全层完成态）

> 形态：与课堂分步一致，每步含【参考答案】与真实执行输出；非聚合脚本。


# AI应用基础 · 工单M2.2《轴承寿命预测系统》
**第7周 | 2课时(80分钟) | 理实一体**  | 教师参考版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 对比LSTM与线性基线的误差曲线，指出LSTM在哪个退化阶段增益最大；示范讲解 |
| **B 组长** | 标准达成 | MSE<0.10；对比≥2模型；解读有业务意义 |
| **C 操作员** | 基础完成 | MSE<0.15；能预测与可视化 |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）

## 一、课前资讯（Predict·预测）
> **预习快问**：窗口长度20、步长1，长度200的序列能切出多少个样本？

> **课前检查**：导学单+预习自测已完成；data/数据已就位、starter可运行（--check环境OK）

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m2rul"], check=True)
print("数据已生成: data/")

工作目录: /Users/joelgong/PycharmProjects/ai-course-teaching-resources/01-AI应用基础


M2.2 -> data/bearing_rul.npz (5400, 20)
完成。数据输出目录: /Users/joelgong/PycharmProjects/ai-course-teaching-resources/01-AI应用基础/data
数据已生成: data/


## 三、任务要求（Make）
**实施步骤**
1. 可视化一个“输入窗口→标签RUL”样例
2. 训练RULNet(≤60 epochs)
3. 计算MSE/RMSE/MAE
4. 绘制真实vs预测RUL曲线+业务解读

**验收标准**：及格(D)=`MSE<0.15；能预测与可视化` ；优秀(B+)=`MSE<0.10；对比≥2模型；解读有业务意义`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [2]:
# -*- coding: utf-8 -*-
"""M2.2 轴承剩余寿命(RUL)预测(LSTM, CPU) —— 学生代码框架"""
import numpy as np
import torch
import torch.nn as nn

DATA = "data/bearing_rul.npz"


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/anaconda3/envs/pp03/lib/python3.12/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/anaconda3/envs/pp03/lib/python3.12/runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "/opt/anaconda3/envs/pp03/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/envs/pp03/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
 

### Starter 2：`class RULNet(nn.Module):`
（Investigate: 读代码→Modify: 补全TODO）

In [3]:
class RULNet(nn.Module):

    def __init__(self, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden,
                            batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):                # x: (N, 20, 1)
        out, _ = self.lstm(x)            # (N,20,H)
        last = out[:, -1, :]             # 取最后时刻 → (N,H)
        return self.fc(last).squeeze(-1)  # (N,)

### Starter 3：`def visualize_window():`
（Investigate: 读代码→Modify: 补全TODO）

In [4]:
def visualize_window():
    """预习用: 看见'输入窗口→标签'的对应关系(运行即出图)"""
    import matplotlib.pyplot as plt
    d = np.load(DATA)
    X, y = d["X"], d["y"]
    i = 100
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(range(i, i + 20), X[i], "o-", label="输入窗口(振动)")
    ax.axvline(i + 20, color="r", ls="--", label=f"预测点RUL={y[i]:.2f}")
    ax.set_xlabel("时间步")
    ax.legend()
    plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS"]
    plt.tight_layout()
    plt.savefig("window_demo.png", dpi=120)
    print("已保存 window_demo.png")

### Starter 4：`def main():`
（Investigate: 读代码→Modify: 补全TODO）

In [5]:
def main():
    d = np.load(DATA)
    X, y = d["X"].astype(np.float32), d["y"].astype(np.float32)
    X = X[:, :, None]                    # (N,20)→(N,20,1)
    n = int(len(X) * 0.8)
    Xtr = torch.tensor(X[:n].tolist(), dtype=torch.float32)
    ytr = torch.tensor(y[:n].tolist(), dtype=torch.float32)
    Xte = torch.tensor(X[n:].tolist(), dtype=torch.float32)
    yte = torch.tensor(y[n:].tolist(), dtype=torch.float32)

    model = RULNet()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    lossf = nn.MSELoss()

    for epoch in range(60):
        model.train()
        opt.zero_grad()
        loss = lossf(model(Xtr), ytr)
        loss.backward()
        opt.step()
        if (epoch + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                mse = lossf(model(Xte), yte).item()
            print(f"epoch {epoch + 1}  训练loss={loss.item():.4f}  验证MSE={mse:.4f}")

    model.eval()
    with torch.no_grad():
        p = model(Xte)
        mse = lossf(p, yte).item()
        rmse = mse ** 0.5
        mae = (p - yte).abs().mean().item()
    print(f"测试指标: MSE={mse:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}")
    print(f"业务解读: RMSE={rmse:.2f}（归一化寿命单位）→ 约可提前 {rmse * 30:.1f} 天窗口介入维护。")

### 💡 分层提示（先自己试5分钟再看）
**思路提示**
> - TODO-1: 全批量(或batch=64)训练, 每10轮打印loss
>   （在此处编写你的代码——依据上方注释/文档字符串）
> - TODO-2: 计算测试集 MSE/RMSE/MAE 三项指标 TODO-3:
>   选1台设备的退化段, 绘制"真实RUL vs 预测RUL"曲线 TODO-4:
>   (对照)线性回归基线: 用窗口均值单特征拟合, 对比MSE TODO-5:
>   (业务解读)≤80字: RMSE=0.1对维护计划意味着什么

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位TODO行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改参数/换实现对比效果，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方"验收标准"，及格(D)=`MSE<0.15；能预测与可视化`，优秀(B+)=`MSE<0.10；对比≥2模型；解读有业务意义`

### Starter 5：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [6]:
if __name__ == "__main__":
    import sys
    if "--viz" in sys.argv:
        visualize_window()
    else:
        main()

epoch 10  训练loss=0.1599  验证MSE=0.1517


epoch 20  训练loss=0.1052  验证MSE=0.1033


epoch 30  训练loss=0.0912  验证MSE=0.0884


epoch 40  训练loss=0.0652  验证MSE=0.0628


epoch 50  训练loss=0.0332  验证MSE=0.0300


epoch 60  训练loss=0.0067  验证MSE=0.0064
测试指标: MSE=0.0064  RMSE=0.0801  MAE=0.0637
业务解读: RMSE=0.08（归一化寿命单位）→ 约可提前 2.4 天窗口介入维护。


## 四、提交要求与避坑
- 提交: `学号_姓名_M2.2.py` 或本notebook(含输出)
- ⚠ 窗口X与标签y错位是头号bug，用--viz可视化核对
- ⚠ CPU训练加早停，epochs≤60观察验证loss
- ⚠ RUL是回归任务，损失用MSELoss而非交叉熵

---

## 教师参考答案（来源：`教师参考/solutions.py` · `m22()`）

> **发放纪律：本附录勿随学生版下发。**

上方学生版 TODO 与下方参考实现的对应关系：

| 学生版函数 | 说明 |
|---|---|
| — | 本课无 TODO |

In [7]:
def m22():
    import torch
    import torch.nn as nn

    d = np.load("data/bearing_rul.npz")
    X = torch.tensor(d["X"][:, :, None].tolist(), dtype=torch.float32)
    y = torch.tensor(d["y"].tolist(), dtype=torch.float32)
    n = int(len(X) * 0.8)
    Xtr, ytr, Xte, yte = X[:n], y[:n], X[n:], y[n:]

    class RULNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.lstm = nn.LSTM(1, 32, batch_first=True)
            self.fc = nn.Linear(32, 1)

        def forward(self, x):
            return self.fc(self.lstm(x)[0][:, -1, :]).squeeze(-1)

    torch.manual_seed(42)
    model = RULNet()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for ep in range(60):
        opt.zero_grad()
        loss = nn.functional.mse_loss(model(Xtr), ytr)
        loss.backward()
        opt.step()
        if (ep + 1) % 10 == 0:
            print(f"epoch{ep + 1} loss={loss.item():.4f}")
    with torch.no_grad():
        p = model(Xte).numpy()
    mse = float(np.mean((p - yte.numpy()) ** 2))
    print(f"MSE={mse:.4f} RMSE={mse ** 0.5:.4f} "
          f"MAE={float(np.mean(np.abs(p - yte.numpy()))):.4f}")
    print("业务解读: RMSE≈0.1即寿命预测误差约10%退化区间, 可提前1-2个维护"
          "周期介入, 满足预测性维护排程需求。")

## 运行与验证

上方参考实现可直接运行（合成数据、CPU 离线、结果可复现）。
多解法对比见《多解法手册.md》。

## ABCD 验收对照

- **D·及格/C·良好**：TODO 分步单元（上方）逐个执行通过，输出即证据（0 个参考答案单元）。
- **B·优秀**：标注「(优秀)」的扩展单元已实现并执行。
- **A·师傅**：多解互验与帮带要点见上节；可用于 2 分钟费曼展示。
